# 03 - Feature Engineering
We engineer features from the past seasons dataset to prepare for model training.
Key features: price efficiency, points per minute, xG metrics, position encoding, lag features.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/player_past_seasons.csv')
print(df.shape)
df.head()

(2017, 37)


,season_name,element_code,start_cost,end_cost,total_points,minutes,goals_scored,assists,clean_sheets,goals_conceded,...,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,player_id,id,name,team,position,price
0,2021/22,154561,45,44,95,2160,0,0,8,27,...,0.00,0.00,0.00,0.00,1,1,Raya,Arsenal,GKP,6.0
1,2022/23,154561,45,48,166,3420,0,0,12,46,...,0.11,0.12,0.23,50.12,1,1,Raya,Arsenal,GKP,6.0
2,2023/24,154561,50,53,135,2880,0,0,16,24,...,0.00,0.04,0.04,22.51,1,1,Raya,Arsenal,GKP,6.0
3,2024/25,154561,55,56,142,3420,0,0,13,34,...,0.00,0.03,0.03,35.03,1,1,Raya,Arsenal,GKP,6.0
4,2025/26,154561,55,62,162,3330,0,0,19,26,...,0.00,0.07,0.07,27.56,1,1,Raya,Arsenal,GKP,6.0


In [2]:
# Convert xG columns to float
xg_cols = ['expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded']
for col in xg_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Price in millions
df['start_price'] = df['start_cost'] / 10
df['end_price'] = df['end_cost'] / 10
df['price_change'] = df['end_price'] - df['start_price']

print('Price columns created')

Price columns created


In [3]:
# Points efficiency features
df['points_per_million'] = df['total_points'] / df['end_price'].clip(lower=0.1)
df['points_per_90'] = df['total_points'] / (df['minutes'].clip(lower=1) / 90)
df['minutes_per_game'] = df['minutes'] / 38  # approx games in a season

# Goal contribution features
df['goal_contributions'] = df['goals_scored'] + df['assists']
df['goals_per_90'] = df['goals_scored'] / (df['minutes'].clip(lower=1) / 90)
df['assists_per_90'] = df['assists'] / (df['minutes'].clip(lower=1) / 90)
df['xg_per_90'] = df['expected_goals'] / (df['minutes'].clip(lower=1) / 90)
df['xa_per_90'] = df['expected_assists'] / (df['minutes'].clip(lower=1) / 90)
df['xgi_per_90'] = df['expected_goal_involvements'] / (df['minutes'].clip(lower=1) / 90)

# Defensive features
df['saves_per_90'] = df['saves'] / (df['minutes'].clip(lower=1) / 90)
df['clean_sheet_rate'] = df['clean_sheets'] / 38

# Bonus and BPS efficiency
df['bonus_per_90'] = df['bonus'] / (df['minutes'].clip(lower=1) / 90)

print('Performance features created')
df[['name','position','points_per_million','points_per_90','xg_per_90','xa_per_90']].head(10)

Performance features created


,name,position,points_per_million,points_per_90,xg_per_90,xa_per_90
0,Raya,GKP,21.590909,3.958333,0.000000,0.000000
1,Raya,GKP,34.583333,4.368421,0.002895,0.003158
2,Raya,GKP,25.471698,4.218750,0.000000,0.001250
3,Raya,GKP,25.357143,3.736842,0.000000,0.000789
4,Raya,GKP,26.129032,4.378378,0.000000,0.001892
5,Arrizabalaga,GKP,26.296296,3.944444,0.000000,0.000000
6,Arrizabalaga,GKP,16.666667,2.727273,0.000000,0.000000
7,Arrizabalaga,GKP,5.531915,4.000000,0.000000,0.000000
8,Arrizabalaga,GKP,4.222222,4.750000,0.000000,0.000000
9,Arrizabalaga,GKP,26.222222,4.140351,0.000000,0.000000


In [4]:
# Lag features - previous season performance
df = df.sort_values(['player_id', 'season_name'])

lag_cols = ['total_points', 'minutes', 'goals_scored', 'assists',
            'clean_sheets', 'bonus', 'points_per_90', 'xg_per_90', 'xa_per_90']

for col in lag_cols:
    df[f'{col}_lag1'] = df.groupby('player_id')[col].shift(1)
    df[f'{col}_lag2'] = df.groupby('player_id')[col].shift(2)

print(f'Lag features created. Shape: {df.shape}')

Lag features created. Shape: (2017, 70)


In [5]:
# Position encoding
pos_dummies = pd.get_dummies(df['position'], prefix='pos')
df = pd.concat([df, pos_dummies], axis=1)

# Season encoding (ordinal)
season_order = {'2021/22': 1, '2022/23': 2, '2023/24': 3, '2024/25': 4, '2025/26': 5}
df['season_num'] = df['season_name'].map(season_order)

print('Encoding done')
print(df.columns.tolist())

Encoding done
['season_name', 'element_code', 'start_cost', 'end_cost', 'total_points', 'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity', 'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries', 'tackles', 'defensive_contribution', 'starts', 'expected_goals', 'expected_assists', 'expected_goal_involvements', 'expected_goals_conceded', 'player_id', 'id', 'name', 'team', 'position', 'price', 'start_price', 'end_price', 'price_change', 'points_per_million', 'points_per_90', 'minutes_per_game', 'goal_contributions', 'goals_per_90', 'assists_per_90', 'xg_per_90', 'xa_per_90', 'xgi_per_90', 'saves_per_90', 'clean_sheet_rate', 'bonus_per_90', 'total_points_lag1', 'total_points_lag2', 'minutes_lag1', 'minutes_lag2', 'goals_scored_lag1', 'goals_scored_lag2', 'assists_lag1', 'assists_lag2', 'clean_sheets_lag1', 'clean_sheets_lag

In [6]:
# Drop rows with NaN lag features (first season per player)
df_model = df.dropna(subset=[f'{c}_lag1' for c in lag_cols])
print(f'Rows before dropna: {len(df)}')
print(f'Rows after dropna: {len(df_model)}')

# Save feature matrix
df_model.to_csv('../data/processed/features.csv', index=False)
print('Feature matrix saved.')
df_model.head()

Rows before dropna: 2017
Rows after dropna: 1519
Feature matrix saved.


,season_name,element_code,start_cost,end_cost,total_points,minutes,goals_scored,assists,clean_sheets,goals_conceded,...,points_per_90_lag2,xg_per_90_lag1,xg_per_90_lag2,xa_per_90_lag1,xa_per_90_lag2,pos_DEF,pos_FWD,pos_GKP,pos_MID,season_num
1,2022/23,154561,45,48,166,3420,0,0,12,46,...,NaN,0.000000,NaN,0.000000,NaN,False,False,True,False,2.0
2,2023/24,154561,50,53,135,2880,0,0,16,24,...,3.958333,0.002895,0.000000,0.003158,0.000000,False,False,True,False,3.0
3,2024/25,154561,55,56,142,3420,0,0,13,34,...,4.368421,0.000000,0.002895,0.001250,0.003158,False,False,True,False,4.0
4,2025/26,154561,55,62,162,3330,0,0,19,26,...,4.218750,0.000000,0.000000,0.000789,0.001250,False,False,True,False,5.0
6,2019/20,109745,55,54,90,2970,0,0,8,47,...,NaN,0.000000,NaN,0.000000,NaN,False,False,True,False,NaN


In [7]:
# Feature importance preview - correlation with target
feature_cols = [c for c in df_model.columns if '_lag1' in c or '_lag2' in c] + \
               ['end_price', 'season_num', 'pos_GKP', 'pos_DEF', 'pos_MID', 'pos_FWD']

feature_cols = [c for c in feature_cols if c in df_model.columns]

corr_with_target = df_model[feature_cols + ['total_points']].corr()['total_points'].drop('total_points').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
corr_with_target.plot(kind='barh', color='steelblue')
plt.title('Feature Correlation with Total Points (Target)')
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.savefig('../data/processed/feature_correlation.png', dpi=150)
plt.show()

print(f'Total features for modelling: {len(feature_cols)}')

<Figure size 1000x800 with 1 Axes>

Total features for modelling: 24
